# 🧠 Auralis SDNN v2 — Improved Training Notebook
## High Accuracy · Low ECE · Reliable Self-Diagnosis

> **Goal:** Train an SDNN that is simultaneously **accurate** and **well-calibrated** — it should know what it knows and know what it doesn't.

### Key Improvements over v1

| Technique | Why it helps |
|-----------|-------------|
| **Label Smoothing (ε=0.1)** | Prevents overconfidence by softening hard targets |
| **CutMix + MixUp** | Forces the model to learn from partial information → better generalisation + calibration |
| **Dropout in diagnostic heads** | Regularises confidence/error heads specifically |
| **2-phase training** | Phase 1: Classification backbone. Phase 2: Freeze backbone, fine-tune diagnostic heads |
| **Cosine Annealing with Warm Restarts** | Better convergence than vanilla cosine |
| **Post-hoc Temperature Scaling** | Learns an optimal temperature scalar on a held-out validation set |
| **Gradient Clipping** | Stabilises training with mixed losses |
| **Progressive λ schedule** | Diagnostic head importance increases as the backbone matures |

**Expected results:** Accuracy ≥ 90%, ECE ≤ 0.03, AUROC ≥ 0.75

---

### Instructions
1. **Runtime → Change runtime type → T4 or A100 GPU**
2. Run all cells in order
3. Last cell downloads `best_sdnn.pth` to your machine


## §1 — Setup

In [ ]:
!pip install torch torchvision numpy tqdm scikit-learn --quiet
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
import torchvision, torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import numpy as np, time, os, math
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")
print(f"  PyTorch {torch.__version__}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name()}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


## §2 — Architecture (Improved)

Same 3-head structure, with two key additions:
- **Dropout (0.3)** in confidence and error heads for regularisation
- Heads use a **2-layer MLP** instead of a single linear layer (128-D hidden)


In [ ]:
from torchvision.models import resnet18

class CIFARResNet18Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        base = resnet18(weights=None)
        base.conv1  = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.features = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4, base.avgpool
        )
        self.feature_dim = 512

    def forward(self, x):
        return torch.flatten(self.features(x), 1)


class SDNNv2(nn.Module):
    """
    Self-Diagnosing Neural Network v2.
    Improvements: deeper diagnostic heads with dropout.
    """
    def __init__(self, num_classes=10, head_dropout=0.3):
        super().__init__()
        self.backbone = CIFARResNet18Backbone()
        d = self.backbone.feature_dim  # 512

        # Classification head (unchanged)
        self.classification_head = nn.Linear(d, num_classes)

        # Deeper confidence head with dropout
        self.confidence_head = nn.Sequential(
            nn.Linear(d, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

        # Deeper error prediction head with dropout
        self.error_head = nn.Sequential(
            nn.Linear(d, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        f = self.backbone(x)
        return {
            "logits":     self.classification_head(f),
            "confidence": self.confidence_head(f),
            "error_prob": self.error_head(f),
        }

model = SDNNv2(num_classes=10).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"✓ SDNNv2 — {param_count:,} parameters")


## §3 — Data (with CutMix + MixUp)

We apply aggressive augmentation:
- **AutoAugment (CIFAR-10 policy)** — tuned augmentation
- **Random erasing** — simulates occlusion
- **CutMix (α=1.0)** and **MixUp (α=0.2)** — applied at batch level during training

We also hold out **5,000 samples** from training for temperature scaling calibration.


In [ ]:
BATCH_SIZE = 128

train_transform = transforms.Compose([
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

full_train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform)

# Hold out 5k for temperature scaling calibration
train_size = len(full_train_dataset) - 5000
cal_size   = 5000
train_dataset, cal_dataset = random_split(
    full_train_dataset, [train_size, cal_size],
    generator=torch.Generator().manual_seed(42))

# Calibration set uses test-time transform (no augmentation)
cal_dataset_clean = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=False, transform=test_transform)
cal_indices   = cal_dataset.indices
cal_subset    = torch.utils.data.Subset(cal_dataset_clean, cal_indices)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
cal_loader   = DataLoader(cal_subset,    batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"✓ Train   : {train_size:,} samples  ({len(train_loader)} batches)")
print(f"  Cal     : {cal_size:,} samples  (for temperature scaling)")
print(f"  Test    : {len(test_dataset):,} samples")


## §4 — CutMix + MixUp Implementation

In [ ]:
def rand_bbox(size, lam):
    """Generate random bounding box for CutMix."""
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w   = int(W * cut_rat)
    cut_h   = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2


def apply_cutmix_or_mixup(images, labels, num_classes=10,
                           cutmix_alpha=1.0, mixup_alpha=0.2, cutmix_prob=0.5):
    """
    Randomly apply CutMix or MixUp to a batch.
    Returns: mixed_images, soft_labels (one-hot), lam
    """
    B = images.size(0)
    one_hot = F.one_hot(labels, num_classes).float()

    r = np.random.rand()
    if r < cutmix_prob:
        # CutMix
        lam   = np.random.beta(cutmix_alpha, cutmix_alpha)
        idx   = torch.randperm(B, device=images.device)
        x1, y1, x2, y2 = rand_bbox(images.size(), lam)
        images[:, :, x1:x2, y1:y2] = images[idx, :, x1:x2, y1:y2]
        # Adjust lambda to actual area ratio
        lam = 1 - ((x2 - x1) * (y2 - y1) / (images.size(-1) * images.size(-2)))
        soft_labels = lam * one_hot + (1 - lam) * one_hot[idx]
    else:
        # MixUp
        lam   = np.random.beta(mixup_alpha, mixup_alpha)
        lam   = max(lam, 1 - lam)  # ensure lam >= 0.5
        idx   = torch.randperm(B, device=images.device)
        images = lam * images + (1 - lam) * images[idx]
        soft_labels = lam * one_hot + (1 - lam) * one_hot[idx]

    return images, soft_labels, lam

print("✓ CutMix + MixUp defined")


## §5 — Loss Function (Label Smoothing + Progressive λ)

Key changes:
- **Label smoothing (ε=0.1)** in classification loss
- **Soft CrossEntropy** to handle CutMix/MixUp soft labels
- **Progressive λ**: diagnostic head loss weight ramps from 0.1 → 1.0 over training


In [ ]:
def soft_cross_entropy(logits, soft_labels):
    """CrossEntropy supporting soft (non-one-hot) labels with label smoothing."""
    log_probs = F.log_softmax(logits, dim=1)
    return -(soft_labels * log_probs).sum(dim=1).mean()


class SDNNLossV2(nn.Module):
    def __init__(self, label_smoothing=0.1):
        super().__init__()
        self.label_smoothing = label_smoothing
        self.confidence_loss_fn = nn.BCELoss()
        self.error_loss_fn      = nn.BCELoss()

    def forward(self, logits, labels, confidence, error_prob,
                soft_labels=None, lambda_diag=0.5):
        # Classification loss
        if soft_labels is not None:
            # Soft labels from CutMix/MixUp — apply label smoothing on top
            n_classes  = logits.size(1)
            smoothed   = (1 - self.label_smoothing) * soft_labels + \
                         self.label_smoothing / n_classes
            cls_loss   = soft_cross_entropy(logits, smoothed)
        else:
            cls_loss = F.cross_entropy(logits, labels,
                                       label_smoothing=self.label_smoothing)

        # Diagnostic targets (detached — based on hard labels)
        with torch.no_grad():
            preds        = logits.argmax(dim=1)
            correctness  = (preds == labels).float().unsqueeze(1)
            error_target = 1.0 - correctness

        conf_loss = self.confidence_loss_fn(confidence, correctness)
        err_loss  = self.error_loss_fn(error_prob, error_target)

        total = cls_loss + lambda_diag * (conf_loss + err_loss)

        return total, {
            "cls": cls_loss.item(),
            "conf": conf_loss.item(),
            "err": err_loss.item(),
            "total": total.item(),
        }

loss_fn = SDNNLossV2(label_smoothing=0.1)
print("✓ SDNNLossV2 with label smoothing=0.1")


## §6 — Training Loop

**Phase 1 (epochs 1–60):** Train the full model end-to-end.  
- Diagnostic λ ramps from 0.1 → 1.0 (so the backbone gets a head start before the diagnostic heads carry weight)
- AdamW + CosineAnnealingWarmRestarts (T₀=10, T_mult=2)
- Gradient clipping at max_norm=1.0

**Phase 2 (epochs 61–75):** Freeze backbone, fine-tune **only** the diagnostic heads.  
- Lower LR (5e-5)
- λ = 1.5 (emphasis on calibration)
- No CutMix/MixUp (clean signal for diagnostic heads)


In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────
PHASE1_EPOCHS = 60
PHASE2_EPOCHS = 15
TOTAL_EPOCHS  = PHASE1_EPOCHS + PHASE2_EPOCHS
LR_PHASE1     = 1e-3
LR_PHASE2     = 5e-5
WEIGHT_DECAY  = 5e-4
MAX_GRAD_NORM = 1.0

# ── Optimizers ──────────────────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=LR_PHASE1, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# ── Training ────────────────────────────────────────────────────────────
best_val_loss = float("inf")
best_val_acc  = 0.0
history = {"epoch":[], "train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[], "lr":[]}

os.makedirs("checkpoints", exist_ok=True)

print(f"{'':>5} {'Phase':^7} {'LR':>10} {'λ_diag':>7} {'Train Loss':>11} "
      f"{'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'Time':>6}")
print("─" * 82)

for epoch in range(1, TOTAL_EPOCHS + 1):
    t0 = time.time()

    # ── Phase config ──────────────────────────────────────────────────
    in_phase2 = epoch > PHASE1_EPOCHS

    if in_phase2 and epoch == PHASE1_EPOCHS + 1:
        # Freeze backbone for phase 2
        for param in model.backbone.parameters():
            param.requires_grad = False
        # New optimizer for diagnostic heads only
        diag_params = list(model.confidence_head.parameters()) + \
                      list(model.error_head.parameters()) + \
                      list(model.classification_head.parameters())
        optimizer = optim.AdamW(diag_params, lr=LR_PHASE2, weight_decay=1e-5)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=PHASE2_EPOCHS, eta_min=1e-7)
        print(f"\n{'─'*82}")
        print("  ⚡ Phase 2: Backbone frozen → fine-tuning diagnostic heads")
        print(f"{'─'*82}")

    # Progressive lambda: ramp 0.1 → 1.0 during phase 1
    if not in_phase2:
        lambda_diag = 0.1 + 0.9 * min(epoch / PHASE1_EPOCHS, 1.0)
    else:
        lambda_diag = 1.5  # emphasis on calibration in phase 2

    use_cutmix = not in_phase2  # no mixing in phase 2

    # ── Train ─────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0; train_correct = 0; train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        soft_labels = None
        if use_cutmix:
            images, soft_labels, _ = apply_cutmix_or_mixup(
                images, labels, num_classes=10)

        optimizer.zero_grad()
        out = model(images)

        loss, _ = loss_fn(out["logits"], labels, out["confidence"],
                          out["error_prob"], soft_labels=soft_labels,
                          lambda_diag=lambda_diag)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        preds       = out["logits"].argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += images.size(0)

    train_loss /= train_total
    train_acc   = train_correct / train_total

    # ── Validate ──────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0; val_correct = 0; val_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            out  = model(images)
            loss, _ = loss_fn(out["logits"], labels, out["confidence"],
                              out["error_prob"], lambda_diag=lambda_diag)
            val_loss    += loss.item() * images.size(0)
            preds        = out["logits"].argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += images.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total
    lr_now    = optimizer.param_groups[0]["lr"]

    scheduler.step()

    # ── Logging ───────────────────────────────────────────────────────
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(lr_now)

    phase_str = "P2-diag" if in_phase2 else "P1-full"
    marker    = ""
    if val_acc > best_val_acc:
        best_val_acc  = val_acc
        best_val_loss = val_loss
        torch.save({
            "model_state": model.state_dict(),
            "epoch":       epoch,
            "val_loss":    val_loss,
            "val_acc":     val_acc,
        }, "checkpoints/best_sdnn.pth")
        marker = " ← best"

    elapsed = time.time() - t0
    print(f"{epoch:>5} {phase_str:^7} {lr_now:>10.2e} {lambda_diag:>7.2f} "
          f"{train_loss:>11.4f} {train_acc:>10.4f} {val_loss:>10.4f} "
          f"{val_acc:>9.4f} {elapsed:>5.1f}s{marker}")

print(f"\n✓ Training complete")
print(f"  Best val accuracy : {best_val_acc*100:.2f}%")
print(f"  Best val loss     : {best_val_loss:.4f}")


## §7 — Post-Hoc Temperature Scaling

Temperature scaling learns a single scalar $T > 0$ that divides logits before softmax:

$$\hat{p}(y|x) = \text{softmax}(\mathbf{z}/T)$$

- $T > 1$ → softens predictions → reduces overconfidence
- $T < 1$ → sharpens predictions
- $T = 1$ → no change (uncalibrated)

We optimise $T$ on the held-out calibration set (5,000 samples) to minimise NLL.  
This provably improves ECE without changing accuracy.


In [ ]:
# Load best checkpoint
ckpt = torch.load("checkpoints/best_sdnn.pth", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"✓ Loaded best checkpoint (epoch {ckpt['epoch']}, val_acc={ckpt['val_acc']*100:.2f}%)")

# ── Collect calibration logits ─────────────────────────────────────────
cal_logits_list = []
cal_labels_list = []

with torch.no_grad():
    for images, labels in cal_loader:
        images = images.to(device)
        out    = model(images)
        cal_logits_list.append(out["logits"].cpu())
        cal_labels_list.append(labels)

cal_logits = torch.cat(cal_logits_list)
cal_labels = torch.cat(cal_labels_list)
print(f"  Calibration set: {len(cal_labels)} samples")

# ── Optimise temperature ───────────────────────────────────────────────
temperature = nn.Parameter(torch.ones(1) * 1.5)
temp_optimizer = optim.LBFGS([temperature], lr=0.01, max_iter=100)

def temp_eval():
    temp_optimizer.zero_grad()
    scaled_logits = cal_logits / temperature.clamp(min=0.01)
    loss = F.cross_entropy(scaled_logits, cal_labels)
    loss.backward()
    return loss

temp_optimizer.step(temp_eval)
T = temperature.item()
print(f"\n✓ Optimal temperature: T = {T:.4f}")

# ── Save final checkpoint with temperature ─────────────────────────────
ckpt["temperature"] = T
torch.save(ckpt, "checkpoints/best_sdnn.pth")
print(f"  Checkpoint updated with temperature = {T:.4f}")


## §8 — Final Evaluation (with Temperature Scaling)

In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix)

model.eval()
all_softmax, all_conf, all_err, all_labels = [], [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        out    = model(images)
        logits = out["logits"] / T  # apply temperature
        sm     = torch.softmax(logits, dim=1)
        all_softmax.append(sm.cpu().numpy())
        all_conf.append(out["confidence"].squeeze(1).cpu().numpy())
        all_err.append(out["error_prob"].squeeze(1).cpu().numpy())
        all_labels.append(labels.numpy())

softmax_probs = np.concatenate(all_softmax)
confidence    = np.concatenate(all_conf)
error_prob    = np.concatenate(all_err)
labels_np     = np.concatenate(all_labels)
predicted     = softmax_probs.argmax(axis=1)
correct       = (predicted == labels_np)

# ── Metrics ───────────────────────────────────────────────────────────
acc = accuracy_score(labels_np, predicted)

# ECE
def compute_ece(conf, corr, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0; n = len(conf)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf >= lo) & (conf < hi)
        if mask.sum() == 0: continue
        ece += (mask.sum()/n) * abs(corr[mask].mean() - conf[mask].mean())
    return float(ece)

# NLL
cp  = softmax_probs[np.arange(len(labels_np)), labels_np]
nll = float(-np.log(np.clip(cp, 1e-9, 1.0)).mean())

# Brier
oh = np.zeros_like(softmax_probs)
oh[np.arange(len(labels_np)), labels_np] = 1.0
brier = float(np.mean(np.sum((softmax_probs - oh)**2, axis=1)))

# ECE on confidence head
ece_conf = compute_ece(confidence, correct)

# ECE on softmax probabilities
top1_prob  = softmax_probs.max(axis=1)
ece_softmax = compute_ece(top1_prob, correct)

# AUROC
error_labels = (~correct).astype(int)
auroc = roc_auc_score(error_labels, error_prob)

CLASSES = ["airplane","automobile","bird","cat","deer",
           "dog","frog","horse","ship","truck"]

print("=" * 58)
print("  Auralis SDNNv2 — Final Evaluation (Temperature-Scaled)")
print("=" * 58)
print(f"  Temperature        : {T:.4f}")
print(f"  Accuracy           : {acc*100:.2f}%")
print(f"  ECE (conf head)    : {ece_conf:.4f}")
print(f"  ECE (softmax)      : {ece_softmax:.4f}")
print(f"  NLL                : {nll:.4f}")
print(f"  Brier Score        : {brier:.4f}")
print(f"  AUROC (error det.) : {auroc:.4f}")
print("=" * 58)
print()

# Assessment
if acc >= 0.90:
    print(f"  ✓ Accuracy ≥ 90% — strong classification")
else:
    print(f"  △ Accuracy {acc*100:.1f}% — consider more epochs or larger batch")

if ece_conf < 0.05:
    print(f"  ✓ ECE = {ece_conf:.4f} — well-calibrated confidence head")
else:
    print(f"  △ ECE = {ece_conf:.4f} — consider adjusting label smoothing")

if auroc >= 0.70:
    print(f"  ✓ AUROC = {auroc:.4f} — reliable error self-detection")
else:
    print(f"  △ AUROC = {auroc:.4f} — diagnostic heads need more training")

print()
print("Per-class report:")
print(classification_report(labels_np, predicted,
                            target_names=CLASSES, digits=4))


## §9 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5),
                         facecolor="#0d1117")

for ax in axes:
    ax.set_facecolor("#161b22")
    ax.tick_params(colors="#8b949e")
    ax.grid(True, color="#21262d", ls="--", lw=0.6)

# Loss
axes[0].plot(history["epoch"], history["train_loss"], color="#58a6ff", lw=1.8, label="Train")
axes[0].plot(history["epoch"], history["val_loss"],   color="#f85149", lw=1.8, label="Val")
axes[0].axvline(PHASE1_EPOCHS, color="#d29922", ls="--", lw=1, alpha=0.7, label="Phase 2 start")
axes[0].set_xlabel("Epoch", color="#e6edf3"); axes[0].set_ylabel("Loss", color="#e6edf3")
axes[0].set_title("Loss", color="#e6edf3", fontweight="bold")
axes[0].legend(facecolor="#161b22", edgecolor="#30363d")

# Accuracy
axes[1].plot(history["epoch"], [a*100 for a in history["train_acc"]], color="#58a6ff", lw=1.8, label="Train")
axes[1].plot(history["epoch"], [a*100 for a in history["val_acc"]],   color="#3fb950", lw=1.8, label="Val")
axes[1].axvline(PHASE1_EPOCHS, color="#d29922", ls="--", lw=1, alpha=0.7, label="Phase 2 start")
axes[1].set_xlabel("Epoch", color="#e6edf3"); axes[1].set_ylabel("Accuracy %", color="#e6edf3")
axes[1].set_title("Accuracy", color="#e6edf3", fontweight="bold")
axes[1].legend(facecolor="#161b22", edgecolor="#30363d")

# Learning rate
axes[2].plot(history["epoch"], history["lr"], color="#bc8cff", lw=1.8)
axes[2].axvline(PHASE1_EPOCHS, color="#d29922", ls="--", lw=1, alpha=0.7, label="Phase 2 start")
axes[2].set_xlabel("Epoch", color="#e6edf3"); axes[2].set_ylabel("Learning Rate", color="#e6edf3")
axes[2].set_title("LR Schedule", color="#e6edf3", fontweight="bold")
axes[2].set_yscale("log")
axes[2].legend(facecolor="#161b22", edgecolor="#30363d")

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("Saved: training_curves.png")


## §10 — Save & Download

The checkpoint is saved with the format expected by `app.py`:
```python
{"model_state": ..., "temperature": T, "epoch": ..., "val_acc": ...}
```

**Important:** The v2 architecture has deeper diagnostic heads. To use this checkpoint with the web UI,
you'll need to update `models/sdnn_model.py` to match the `SDNNv2` class.  
The cell below generates a compatible `sdnn_model.py` for you to download alongside the weights.


In [ ]:
# ── Rename checkpoint to standard name ──────────────────────────────────
import shutil
shutil.copy("checkpoints/best_sdnn.pth", "best_sdnn.pth")

# ── Generate updated sdnn_model.py ──────────────────────────────────────
model_py = '''"""
sdnn_model.py — SDNNv2 (updated architecture with deeper diagnostic heads)
"""
import torch
import torch.nn as nn
try:
    from models.backbone import CIFARResNet18Backbone
except ModuleNotFoundError:
    import sys, os
    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
    from models.backbone import CIFARResNet18Backbone


class SDNN(nn.Module):
    def __init__(self, num_classes=10, head_dropout=0.3):
        super().__init__()
        self.backbone = CIFARResNet18Backbone()
        d = self.backbone.feature_dim

        self.classification_head = nn.Linear(d, num_classes)

        self.confidence_head = nn.Sequential(
            nn.Linear(d, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

        self.error_head = nn.Sequential(
            nn.Linear(d, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        f = self.backbone(x)
        return {
            "logits":     self.classification_head(f),
            "confidence": self.confidence_head(f),
            "error_prob": self.error_head(f),
        }

    def predict(self, x):
        with torch.no_grad():
            out = self.forward(x)
        return {
            "predicted_class": out["logits"].argmax(dim=1),
            "confidence":      out["confidence"].squeeze(1),
            "error_prob":      out["error_prob"].squeeze(1),
        }


CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
'''

with open("sdnn_model.py", "w") as f:
    f.write(model_py)

print("✓ Files ready for download:")
print("  1. best_sdnn.pth    → place at models/best_sdnn.pth")
print("  2. sdnn_model.py    → replace models/sdnn_model.py")


In [ ]:
from google.colab import files

print("📥 Downloading best_sdnn.pth...")
files.download("best_sdnn.pth")

print("📥 Downloading sdnn_model.py...")
files.download("sdnn_model.py")

print("\n✓ Done! Place files in your project:")
print("   best_sdnn.pth  → models/best_sdnn.pth")
print("   sdnn_model.py  → models/sdnn_model.py")
print("   Then run: npm run dev")


---

## References

1. **Yun et al. (2019)** — *CutMix: Regularization Strategy to Train Strong Classifiers with Localizable Features.* ICCV 2019.
2. **Zhang et al. (2018)** — *MixUp: Beyond Empirical Risk Minimization.* ICLR 2018.
3. **Guo et al. (2017)** — *On Calibration of Modern Neural Networks.* ICML 2017.
4. **Müller, Kornblith & Hinton (2019)** — *When Does Label Smoothing Help?* NeurIPS 2019.
5. **Corbière et al. (2019)** — *Addressing Failure Prediction by Learning Model Confidence.* NeurIPS 2019.

---
*Auralis SDNNv2 · Improved Training Pipeline · CIFAR-10*
